In [63]:
from pyspark.sql.functions import lit, concat, count, desc, regexp_replace, trim, col, lower, split, explode
from pyspark.sql.functions import length
from collections.abc import Iterable
import re

StatementMeta(, 36a09944-5f71-4af3-966b-af3b516035a7, 72, Finished, Available, Finished, False)

In [42]:
words_df = spark.createDataFrame([('cat',), ('elephant',), ('rat',), ('rat',), ('cat', )], ['word'])

words_df.show()

print (type(words_df))

words_df.printSchema()

StatementMeta(, 36a09944-5f71-4af3-966b-af3b516035a7, 48, Finished, Available, Finished, False)

+--------+
|    word|
+--------+
|     cat|
|elephant|
|     rat|
|     rat|
|     cat|
+--------+

<class 'pyspark.sql.dataframe.DataFrame'>
root
 |-- word: string (nullable = true)



In [43]:
words_df.toPandas().head()

StatementMeta(, 36a09944-5f71-4af3-966b-af3b516035a7, 49, Finished, Available, Finished, False)

,word
0,cat
1,elephant
2,rat
3,rat
4,cat


In [44]:
display(words_df.limit(5))

StatementMeta(, 36a09944-5f71-4af3-966b-af3b516035a7, 50, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e94334ee-a13f-4dfc-9553-295ee6046cac)

In [45]:
plural_df = words_df.select(concat(words_df.word, lit('s')).alias('word'))

plural_df.show()
plural_df.limit(5).toPandas().head()

StatementMeta(, 36a09944-5f71-4af3-966b-af3b516035a7, 51, Finished, Available, Finished, False)

+---------+
|     word|
+---------+
|     cats|
|elephants|
|     rats|
|     rats|
|     cats|
+---------+



,word
0,cats
1,elephants
2,rats
3,rats
4,cats


In [46]:
plural_lenght_df = plural_df.select(length('word'))

plural_lenght_df.show()
plural_lenght_df.limit(5).toPandas().head()

StatementMeta(, 36a09944-5f71-4af3-966b-af3b516035a7, 52, Finished, Available, Finished, False)

+------------+
|length(word)|
+------------+
|           4|
|           9|
|           4|
|           4|
|           4|
+------------+



,length(word)
0,4
1,9
2,4
3,4
4,4


In [47]:
as_selft = lambda v: map(lambda r: r[0] if isinstance(r, Iterable) and len(r) == 1 else r, v)

set(as_selft(plural_lenght_df.collect()))

StatementMeta(, 36a09944-5f71-4af3-966b-af3b516035a7, 53, Finished, Available, Finished, False)

{4, 9}

In [48]:
word_count_df = words_df.groupBy('word').count().sort(desc('word'))

word_count_df.show()

StatementMeta(, 36a09944-5f71-4af3-966b-af3b516035a7, 54, Finished, Available, Finished, False)

+--------+-----+
|    word|count|
+--------+-----+
|     rat|    2|
|elephant|    1|
|     cat|    2|
+--------+-----+



In [49]:
set(word_count_df.collect())

StatementMeta(, 36a09944-5f71-4af3-966b-af3b516035a7, 55, Finished, Available, Finished, False)

{Row(word='cat', count=2),
 Row(word='elephant', count=1),
 Row(word='rat', count=2)}

In [50]:
(word_count_df.groupBy().mean().first()['avg(count)'])

StatementMeta(, 36a09944-5f71-4af3-966b-af3b516035a7, 56, Finished, Available, Finished, False)

1.6666666666666667

In [51]:
def wordCount(wordListDF):
    return wordListDF.groupBy('word').count().sort(desc('word'))


wordCount(words_df).show()

StatementMeta(, 36a09944-5f71-4af3-966b-af3b516035a7, 57, Finished, Available, Finished, False)

+--------+-----+
|    word|count|
+--------+-----+
|     rat|    2|
|elephant|    1|
|     cat|    2|
+--------+-----+



In [52]:
[(row[0], row[1])for row in wordCount(words_df).collect()]

StatementMeta(, 36a09944-5f71-4af3-966b-af3b516035a7, 58, Finished, Available, Finished, False)

[('rat', 2), ('elephant', 1), ('cat', 2)]

In [53]:
def removePunctuation(column):
    return trim(regexp_replace(lower(column), '[^a-zA-Z0-9 ]', ''))


sentence_df = spark.createDataFrame([('Hi, you!',),
                                         (' No under_score!',),
                                         (' *      Remove punctuation then spaces  * ',)], ['sentence'])
sentence_df.show(truncate=False)

(sentence_df.select(removePunctuation(col('sentence'))).show(truncate=False))

StatementMeta(, 36a09944-5f71-4af3-966b-af3b516035a7, 59, Finished, Available, Finished, False)

+------------------------------------------+
|sentence                                  |
+------------------------------------------+
|Hi, you!                                  |
| No under_score!                          |
| *      Remove punctuation then spaces  * |
+------------------------------------------+

+---------------------------------------------------------+
|trim(regexp_replace(lower(sentence), [^a-zA-Z0-9 ], , 1))|
+---------------------------------------------------------+
|hi you                                                   |
|no underscore                                            |
|remove punctuation then spaces                           |
+---------------------------------------------------------+



In [65]:
path = 'abfss://6a4c36dd-3670-49dd-8002-7a85654d3f49@onelake.dfs.fabric.microsoft.com/d40a3bf5-77a8-4b37-8a4f-0da9cd8edbb5/Files/shakespeare.txt'
shakespeareDF = spark.read.text(path).select(removePunctuation(col('value')).alias('text'))

shakespeareDF.show(15, truncate=False)

StatementMeta(, 36a09944-5f71-4af3-966b-af3b516035a7, 74, Finished, Available, Finished, False)

+-----------------------------------------------------------------------------------+
|text                                                                               |
+-----------------------------------------------------------------------------------+
|project gutenbergs the complete works of william shakespeare by william shakespeare|
|                                                                                   |
|this ebook is for the use of anyone anywhere in the united states and              |
|most other parts of the world at no cost and with almost no restrictions           |
|whatsoever  you may copy it give it away or reuse it under the terms               |
|of the project gutenberg license included with this ebook or online at             |
|wwwgutenbergorg  if you are not located in the united states youll                 |
|have to check the laws of the country where you are located before using           |
|this ebook                                           

In [80]:
shakespeareDF.limit(10).toPandas().head()

StatementMeta(, 36a09944-5f71-4af3-966b-af3b516035a7, 89, Finished, Available, Finished, False)

,text
0,project gutenbergs the complete works of willi...
1,
2,this ebook is for the use of anyone anywhere i...
3,most other parts of the world at no cost and w...
4,whatsoever you may copy it give it away or re...


In [82]:
shakeWordsDF = shakespeareDF.select(explode(split(col('text'), '\s+')).alias('word'))

shakeWordsDF = shakeWordsDF.filter(col('word') != '')
shakeWordsDF.show()

shakeWordsDFCount = shakeWordsDF.count()
print(shakeWordsDFCount)

StatementMeta(, 36a09944-5f71-4af3-966b-af3b516035a7, 154, Finished, Available, Finished, False)

+-----------+
|       word|
+-----------+
|    project|
| gutenbergs|
|        the|
|   complete|
|      works|
|         of|
|    william|
|shakespeare|
|         by|
|    william|
|shakespeare|
|       this|
|      ebook|
|         is|
|        for|
|        the|
|        use|
|         of|
|     anyone|
|   anywhere|
+-----------+
only showing top 20 rows

961306


In [83]:
topWordsAndCountsDF = wordCount(shakeWordsDF).sort(desc('count'))
topWordsAndCountsDF.show()

StatementMeta(, 36a09944-5f71-4af3-966b-af3b516035a7, 157, Finished, Available, Finished, False)

+----+-----+
|word|count|
+----+-----+
| the|30205|
| and|28386|
|   i|21949|
|  to|20923|
|  of|18822|
|   a|16182|
| you|14437|
|  my|13180|
|  in|12232|
|that|11776|
|  is| 9713|
| not| 9066|
|with| 8528|
|  me| 8263|
| for| 8195|
|  it| 8180|
| his| 7581|
|  be| 7370|
|this| 7178|
|your| 7076|
+----+-----+
only showing top 20 rows

